#<font color="Green">**Notebook Purpose**</font>

This notebook constructs two comparison tables that quantify whether trajectory-eligible patients differ systematically from trajectory-ineligible patients within the primary analytic cohort. This directly addresses Reviewer 1 Comment 2's request for "a comparison of included versus excluded patients for the longitudinal outcome models."

**Eligibility rule** (per `ClusterLevelPlotsCreation.ipynb`, Cell 12): A patient is trajectory-eligible for a given variable (HbA1c or BMI) if they have at least one recorded measurement in **every** calendar year from 2019 through 2024. Assessed independently per variable.

**Outputs:**
- **eTable 5** — Primary cohort patients eligible vs. ineligible for HbA1c trajectory analysis
- **eTable 6** — Primary cohort patients eligible vs. ineligible for BMI trajectory analysis

Both tables follow the same demographic and baseline clinical structure as eTable 4, with standardized mean differences (SMDs) used to quantify group differences. SMDs are preferred over p-values because with samples in the thousands per group, trivially small differences become statistically significant.

**SMD interpretation:** < 0.10 = negligible, 0.10–0.20 = small, > 0.20 = meaningful

---

###<font color="Red"> Required Data </font>

1. **`medication_info.csv`** — `patient_id`, `start_date`, `ingredient`, `medication_class`
2. **`lab_results.csv`** — `patient_id`, `date`, `lab_result_num_val` (HbA1c, LOINC-filtered, cleaned to 3.5–20%)
3. **`BMI_vital_signs.csv`** — `patient_id`, `date`, `value` (BMI)
4. **`patient_demographics.csv`** — `patient_id`, `sex`, `race/ethnicity`, `year_of_birth`, `patient_regional_location`
5. **`patient_comorbidities.csv`** — `patient_id`, `date`, `HF`, `CKD`
6. **`baseline_a1c.csv`** — output from `get_closest_continuous_value_after_t0` for primary cohort
7. **`baseline_bmi.csv`** — output from `get_closest_continuous_value_after_t0` for primary cohort
8. **`early_dropout_patients.pkl`** — dict `{cluster_id: [patient_ids]}` for early-disengagement clusters

All baseline CSVs are reused from the existing Table 1 workflow — no new external function calls are required.

## Section 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl

In [ ]:
medication_info      = pd.read_csv('/content/medication_info.csv')
lab_results          = pd.read_csv('/content/lab_results.csv')
BMI_vital_signs      = pd.read_csv('/content/BMI_vital_signs.csv')
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_comorbidities= pd.read_csv('/content/patient_comorbidities.csv')
baseline_a1c         = pd.read_csv('/content/baseline_a1c_all.csv')
baseline_bmi         = pd.read_csv('/content/baseline_bmi_all.csv')

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pkl.load(f)

# Parse dates
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'], errors='coerce')
lab_results['date']           = pd.to_datetime(lab_results['date'],           errors='coerce')
BMI_vital_signs['date']       = pd.to_datetime(BMI_vital_signs['date'],       errors='coerce')
patient_comorbidities['date'] = pd.to_datetime(patient_comorbidities['date'], errors='coerce')

# Ensure consistent patient_id dtype throughout
for df in [medication_info, lab_results, BMI_vital_signs, patient_demographics,
           patient_comorbidities, baseline_a1c, baseline_bmi]:
    df['patient_id'] = df['patient_id'].astype(str)

print('Data loaded.')
print(f'  Primary cohort size in patient_demographics: {patient_demographics["patient_id"].nunique():,}')

## Section 2 — Restrict to Primary Analytic Cohort

We exclude early-disengagement patients to focus on the 9,327 patients whose trajectories were actually modeled in the manuscript.

In [ ]:
# Flatten dropout IDs to a set
dropout_ids = set()
for _, id_list in early_dropout_patients.items():
    dropout_ids.update([str(x) for x in id_list])

# All primary cohort patient_ids (anyone in demographics who is NOT a dropout)
all_demographics_ids = set(patient_demographics['patient_id'])
primary_cohort_ids   = all_demographics_ids - dropout_ids

N_PRIMARY = len(primary_cohort_ids)
print(f'Primary cohort: {N_PRIMARY:,} (expect 9,327)')

## Section 3 — Build Trajectory Eligibility Flags

For each patient in the primary cohort, determine whether they have at least one recorded measurement in **every** year from 2019 through 2024. HbA1c eligibility and BMI eligibility are assessed independently.

In [ ]:
REQUIRED_YEARS = list(range(2019, 2025))  # [2019, 2020, 2021, 2022, 2023, 2024]

def patients_with_value_every_year(value_df, date_col='date'):
    """
    Return the set of patient_ids that have at least one row of `value_df`
    in every required year (2019-2024). Patient_id dtype must already be str.
    """
    df = value_df[[ 'patient_id', date_col ]].dropna(subset=[date_col]).copy()
    df['year'] = df[date_col].dt.year
    df = df[df['year'].isin(REQUIRED_YEARS)]

    # Count distinct years per patient
    yrs_per_patient = df.groupby('patient_id')['year'].nunique()
    eligible_ids = set(yrs_per_patient[yrs_per_patient == len(REQUIRED_YEARS)].index)
    return eligible_ids

hba1c_eligible_ids = patients_with_value_every_year(lab_results,     date_col='date')
bmi_eligible_ids   = patients_with_value_every_year(BMI_vital_signs, date_col='date')

# Restrict to primary cohort
hba1c_eligible_primary   = hba1c_eligible_ids & primary_cohort_ids
hba1c_ineligible_primary = primary_cohort_ids - hba1c_eligible_primary

bmi_eligible_primary     = bmi_eligible_ids & primary_cohort_ids
bmi_ineligible_primary   = primary_cohort_ids - bmi_eligible_primary

print('HbA1c trajectory eligibility (primary cohort):')
print(f'  Eligible:   {len(hba1c_eligible_primary):,} ({len(hba1c_eligible_primary)/N_PRIMARY*100:.1f}%)')
print(f'  Ineligible: {len(hba1c_ineligible_primary):,} ({len(hba1c_ineligible_primary)/N_PRIMARY*100:.1f}%)')
print()
print('BMI trajectory eligibility (primary cohort):')
print(f'  Eligible:   {len(bmi_eligible_primary):,} ({len(bmi_eligible_primary)/N_PRIMARY*100:.1f}%)')
print(f'  Ineligible: {len(bmi_ineligible_primary):,} ({len(bmi_ineligible_primary)/N_PRIMARY*100:.1f}%)')

## Section 4 — Configuration & Helper Functions

Reuse the variable definitions and stat-computation logic from `ComparisonTable_Primary_vs_Disengagement.ipynb` (eTable 4). Output schema is identical so the new tables are visually consistent with eTable 4.

In [ ]:
# =========================
# Variable definitions
# =========================
RACE_INPUT_LEVELS = ['White', 'Hispanic', 'Asian', 'Black', 'Other']
RACE_OUTPUT_MAP = {
    'Black':    'African American/Black, NH',
    'Asian':    'Asian, NH',
    'Hispanic': 'Hispanic/Latinx',
    'White':    'White, NH',
    'Other':    'Other, NH',
}
REGION_LEVELS = ['Midwest', 'South', 'West', 'Northeast']

AGE_BINS   = [-np.inf, 45, 55, 65, np.inf]
AGE_LABELS = ['- 18-45', '- 45-54', '- 55-64', '- 65+']

A1C_BINS   = [-np.inf, 6.4, 8.0, 9.0, np.inf]
A1C_LABELS = ['- <6.4', '- 6.5-7.9', '- 8.0-8.9', '- >= 9.0']

BMI_BINS   = [-np.inf, 24.9, 29.9, 34.9, 39.9, np.inf]
BMI_LABELS = ['- <24.9', '- 25.0-29.9', '- 30.0-34.9', '- 35.0-39.9', '- >= 40']

WINDOW_END = pd.Timestamp('2019-06-01')

In [ ]:
# =========================
# Per-group statistics function (adapted from ComparisonTable notebook)
# =========================

def compute_group_stats(group_ids, dem_df, a1c_df, bmi_df, com_df):
    """
    Compute all Table 1 statistics for an arbitrary subset of patient_ids.
    Returns a dict {label: value} of counts, means, and SDs.
    """
    dem_g = dem_df[dem_df['patient_id'].isin(group_ids)].copy()
    a1c_g = a1c_df[a1c_df['patient_id'].isin(group_ids)].copy()
    bmi_g = bmi_df[bmi_df['patient_id'].isin(group_ids)].copy()
    com_g = com_df[com_df['patient_id'].isin(group_ids)].copy()

    n = len(dem_g)
    stats = {'N': n}

    # --- Female sex ---
    stats['Female sex'] = int(
        dem_g['sex'].astype(str).str.upper().str.strip()
        .isin(['F', 'FEMALE']).sum()
    )

    # --- Age ---
    dem_g['age_2019'] = 2019 - pd.to_numeric(dem_g['year_of_birth'], errors='coerce')
    age_cuts = pd.cut(dem_g['age_2019'], bins=AGE_BINS,
                      labels=[l.replace('- ', '') for l in AGE_LABELS], right=False)
    age_cnt = age_cuts.value_counts()
    for edge_lbl, display_lbl in zip(age_cnt.index.categories, AGE_LABELS):
        stats[display_lbl] = int(age_cnt.get(edge_lbl, 0))
    stats['age_mean'] = dem_g['age_2019'].mean()
    stats['age_std']  = dem_g['age_2019'].std()

    # --- Race/ethnicity ---
    race_series = dem_g['race/ethnicity'].astype(str).str.strip()
    for in_lbl in RACE_INPUT_LEVELS:
        out_lbl = RACE_OUTPUT_MAP[in_lbl]
        stats[f'- {out_lbl}'] = int((race_series == in_lbl).sum())

    # --- Region ---
    region_series = dem_g['patient_regional_location'].astype(str).str.strip()
    for r in REGION_LEVELS:
        stats[f'- {r}'] = int((region_series == r).sum())

    # --- Baseline HbA1c ---
    a1c_vals = pd.to_numeric(a1c_g['baseline_a1c_value'], errors='coerce')
    stats['a1c_missing'] = int(n - a1c_vals.notna().sum())  # missing relative to group total
    a1c_valid = a1c_vals.dropna()
    stats['a1c_mean'] = a1c_valid.mean() if len(a1c_valid) else np.nan
    stats['a1c_std']  = a1c_valid.std()  if len(a1c_valid) else np.nan
    a1c_binned = pd.cut(a1c_valid, bins=A1C_BINS,
                        labels=[l.replace('- ', '') for l in A1C_LABELS], right=False)
    a1c_cnt = a1c_binned.value_counts()
    for edge_lbl, display_lbl in zip(a1c_cnt.index.categories, A1C_LABELS):
        stats[display_lbl] = int(a1c_cnt.get(edge_lbl, 0))

    # --- Baseline BMI ---
    bmi_vals = pd.to_numeric(bmi_g['baseline_bmi_value'], errors='coerce')
    stats['bmi_missing'] = int(n - bmi_vals.notna().sum())
    bmi_valid = bmi_vals.dropna()
    stats['bmi_mean'] = bmi_valid.mean() if len(bmi_valid) else np.nan
    stats['bmi_std']  = bmi_valid.std()  if len(bmi_valid) else np.nan
    bmi_binned = pd.cut(bmi_valid, bins=BMI_BINS,
                        labels=[l.replace('- ', '') for l in BMI_LABELS], right=True)
    bmi_cnt = bmi_binned.value_counts()
    for edge_lbl, display_lbl in zip(bmi_cnt.index.categories, BMI_LABELS):
        stats[display_lbl] = int(bmi_cnt.get(edge_lbl, 0))

    # --- Comorbidities (anytime up to June 1, 2019) ---
    win = com_g[com_g['date'] <= WINDOW_END].copy()
    for col in ['HF', 'CKD']:
        win[col] = win[col].astype(str).str.strip().str.upper().isin(['TRUE', '1', 'T', 'Y', 'YES'])
    stats['HF']  = int(win.loc[win['HF'],  'patient_id'].nunique())
    stats['CKD'] = int(win.loc[win['CKD'], 'patient_id'].nunique())

    return stats

In [ ]:
# =========================
# SMD computation
# =========================

def smd_proportion(p1, n1, p2, n2):
    """Standardized mean difference for a binary/proportion variable."""
    if n1 == 0 or n2 == 0:
        return np.nan
    pooled_sd = np.sqrt((p1*(1-p1) + p2*(1-p2)) / 2)
    if pooled_sd == 0:
        return 0.0
    return abs(p1 - p2) / pooled_sd

def smd_continuous(m1, s1, m2, s2):
    """Standardized mean difference for a continuous variable."""
    if pd.isna(s1) or pd.isna(s2):
        return np.nan
    pooled_sd = np.sqrt((s1**2 + s2**2) / 2)
    if pooled_sd == 0:
        return 0.0
    return abs(m1 - m2) / pooled_sd

## Section 5 — Assemble Comparison Table Function

A single function that takes two patient_id sets (eligible, ineligible) plus group labels and returns a finished comparison DataFrame with SMDs.

In [ ]:
def fmt_count_pct(n, total):
    if total == 0:
        return 'n=0, 0.0%'
    return f'n={int(n):,}, {n/total*100:.1f}%'

def fmt_mean_sd(m, s):
    if pd.isna(m) or pd.isna(s):
        return ''
    return f'{m:.1f} ± {s:.1f}'

def build_comparison_table(eligible_ids, ineligible_ids,
                           dem_df, a1c_df, bmi_df, com_df,
                           label_eligible='Eligible', label_ineligible='Ineligible'):
    s_e = compute_group_stats(eligible_ids,   dem_df, a1c_df, bmi_df, com_df)
    s_i = compute_group_stats(ineligible_ids, dem_df, a1c_df, bmi_df, com_df)

    n_e, n_i = s_e['N'], s_i['N']
    rows = []

    def add_count(label, key, denom_e=None, denom_i=None):
        de = n_e if denom_e is None else denom_e
        di = n_i if denom_i is None else denom_i
        p_e = s_e[key] / de if de else 0.0
        p_i = s_i[key] / di if di else 0.0
        rows.append((label, fmt_count_pct(s_e[key], de),
                     fmt_count_pct(s_i[key], di),
                     f'{smd_proportion(p_e, de, p_i, di):.3f}'))

    def add_continuous(label, mean_key, sd_key):
        smd = smd_continuous(s_e[mean_key], s_e[sd_key], s_i[mean_key], s_i[sd_key])
        rows.append((label, fmt_mean_sd(s_e[mean_key], s_e[sd_key]),
                     fmt_mean_sd(s_i[mean_key], s_i[sd_key]),
                     f'{smd:.3f}'))

    def add_header(label):
        rows.append((label, '', '', ''))

    # N
    rows.append(('N', f'{n_e:,}', f'{n_i:,}', ''))

    add_header('Demographics')
    add_count('- Female sex', 'Female sex')
    add_continuous('Age (years), mean ± SD', 'age_mean', 'age_std')
    add_header('Age category, years')
    for lbl in AGE_LABELS:
        add_count(lbl, lbl)

    add_header('Race/ethnicity')
    for in_lbl in RACE_INPUT_LEVELS:
        out_lbl = RACE_OUTPUT_MAP[in_lbl]
        add_count(f'- {out_lbl}', f'- {out_lbl}')

    add_header('US Region')
    for r in REGION_LEVELS:
        add_count(f'- {r}', f'- {r}')

    add_header('Clinical Characteristics')
    add_header('Baseline HbA1c (pre-treatment, 2019)')
    add_continuous('  Mean ± SD (%)', 'a1c_mean', 'a1c_std')
    for lbl in A1C_LABELS:
        add_count(lbl, lbl)
    add_count('- Missing', 'a1c_missing')

    add_header('Baseline BMI (pre-treatment, 2019, kg/m²)')
    add_continuous('  Mean ± SD (kg/m²)', 'bmi_mean', 'bmi_std')
    for lbl in BMI_LABELS:
        add_count(lbl, lbl)
    add_count('- Missing', 'bmi_missing')

    add_header('Comorbidities before June 1, 2019')
    add_count('- Heart Failure', 'HF')
    add_count('- Chronic Kidney Disease', 'CKD')

    df = pd.DataFrame(rows, columns=['Characteristic',
                                     f'{label_eligible} (n={n_e:,})',
                                     f'{label_ineligible} (n={n_i:,})',
                                     'SMD'])
    return df

## Section 6 — Build eTable 5 (HbA1c Trajectory Eligibility)

In [ ]:
etable5 = build_comparison_table(
    eligible_ids   = hba1c_eligible_primary,
    ineligible_ids = hba1c_ineligible_primary,
    dem_df = patient_demographics,
    a1c_df = baseline_a1c,
    bmi_df = baseline_bmi,
    com_df = patient_comorbidities,
    label_eligible   = 'HbA1c-Eligible',
    label_ineligible = 'HbA1c-Ineligible',
)
etable5

## Section 7 — Build eTable 6 (BMI Trajectory Eligibility)

In [ ]:
etable6 = build_comparison_table(
    eligible_ids   = bmi_eligible_primary,
    ineligible_ids = bmi_ineligible_primary,
    dem_df = patient_demographics,
    a1c_df = baseline_a1c,
    bmi_df = baseline_bmi,
    com_df = patient_comorbidities,
    label_eligible   = 'BMI-Eligible',
    label_ineligible = 'BMI-Ineligible',
)
etable6

## Section 8 — Export

In [ ]:
!pip install xlsxwriter

In [ ]:
OUT_XLSX = '/content/Trajectory_Eligibility_Comparison.xlsx'

with pd.ExcelWriter(OUT_XLSX, engine='xlsxwriter') as writer:
    etable5.to_excel(writer, sheet_name='eTable 5 - HbA1c', index=False)
    etable6.to_excel(writer, sheet_name='eTable 6 - BMI',   index=False)

    for sheet in ['eTable 5 - HbA1c', 'eTable 6 - BMI']:
        ws = writer.sheets[sheet]
        ws.set_column(0, 0, 45)
        ws.set_column(1, 3, 22)

print(f'Exported to: {OUT_XLSX}')

## Section 9 — Interpretation Notes for Response Letter

**Suggested footnotes (one per table):**

> *eTable 5. Demographic and baseline clinical characteristics of primary-cohort patients eligible vs. ineligible for HbA1c trajectory analysis. Eligibility required at least one recorded HbA1c value in every calendar year from 2019 through 2024. Standardized mean differences (SMD) quantify the magnitude of difference between groups independent of sample size. Baseline HbA1c and BMI represent the closest recorded value at or before each patient's first GLM prescription within 2019; 'Missing' indicates patients without a qualifying pre-treatment value.*

> *eTable 6. Demographic and baseline clinical characteristics of primary-cohort patients eligible vs. ineligible for BMI trajectory analysis. Eligibility required at least one recorded BMI value in every calendar year from 2019 through 2024. Other notes as for eTable 5.*

**What to look for when interpreting:**

- **All SMDs < 0.10:** trajectory-eligible and ineligible patients are well-balanced on the variable. The complete-case analysis approximates the full-cohort target.
- **SMDs 0.10–0.20:** small imbalance; worth noting but unlikely to materially alter conclusions.
- **SMDs > 0.20:** meaningful imbalance. Direction matters: if the ineligible group has higher comorbidity burden or different racial/ethnic composition, this should be acknowledged in Limitations as a potential source of bias in trajectory-level findings.

**Expected based on the Comment 1 missingness check:**
BMI eligibility will be substantially more restrictive than HbA1c eligibility (43.0% of the cohort has no 2019 BMI at all). The BMI-ineligible group will be much larger, and SMDs may be correspondingly larger. This is a real signal — likely reflecting which TriNetX-participating sites contribute vital signs consistently — not an artifact, and the response letter should acknowledge it honestly rather than soft-pedal.